# 표준 Notebook 샘플 - Iris 품질 분류

MLOps 코어모듈에 재학습 파이프라인을 등록할 때 따르는 표준 구조.
`mlops` SDK로 데이터/모델 자산을 받고, 학습하고, 결과를 다시 자산으로 등록한다.
물리 경로(로컬 폴더, S3 URI)는 어디에도 등장하지 않는다.

In [ ]:
# Papermill Parameters
# (MLOps Core의 Generic Airflow Executor가 실행 시점에 override 한다)
output_dir = "./output"
mlflow_tracking_uri = "http://mlflow:5000"
dataset_key = "demo.iris"
dataset_version = "v1"
model_key = "demo.iris_model"


### ① MLOps SDK Import

In [ ]:
import os
import prizm
import pandas as pd
import joblib
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


### ② Data Asset 다운로드
물리 경로를 몰라도 된다. Key + Version만 알면 된다.

In [ ]:
dataset_path = prizm.dataset.download(dataset_key, dataset_version)
df = pd.read_csv(dataset_path)
df.head()


### ③ Hyperparameter

In [ ]:
test_size = 0.2
random_state = 42
max_iter = 200


### ④ Data Split

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state
)


### ⑤ Training

In [ ]:
clf = LogisticRegression(max_iter=max_iter)
clf.fit(X_train, y_train)


### ⑥ Evaluation

In [ ]:
acc = accuracy_score(y_test, clf.predict(X_test))
print(f"accuracy: {acc:.4f}")


### ⑦ Model Save / Asset 등록
결과물을 로컬에 방치하지 않고 `output_dir`(계약된 출력 위치)에 저장한 뒤,
`prizm.model.upload()`로 AI Hub에 등록한다. 동시에 MLflow에 Run/Metric을 남긴다.

In [ ]:
os.makedirs(output_dir, exist_ok=True)
model_path = os.path.join(output_dir, "model.joblib")
joblib.dump(clf, model_path)

mlflow.set_tracking_uri(mlflow_tracking_uri)
mlflow.set_experiment("mlops-core-demo")

with mlflow.start_run(run_name="iris-training"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", max_iter)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_artifact(model_path)

    result = prizm.model.upload(model_key, model_path)
    mlflow.set_tag("mlops_asset_key", result["key"])
    mlflow.set_tag("mlops_asset_version", result["version"])

print("registered:", result)
